# 📝 교안_03 과제: 관측성과 디버깅

> 교안_03 에서 배운 것을 **헬스장 회원권** 도메인으로 한 번씩 훈련합니다. 콜백 부착 · 토큰 집계 · 비용 계산 · 서버 값 대조 · 프롬프트 버전 관리 · 세션 · 라우팅 고치기.

## 푸는 방법
1. 맨 위 **준비 셀들**을 먼저 실행하세요(관측 콜백·서버 조회 도우미·도구가 제공됩니다).
2. 각 문제의 **답안 셀**을 채우고 바로 아래 **자가채점 셀**로 확인하세요.

**이 과제는 실제 호출과 실제 전송을 합니다.** `.env` 에 `OPENAI_API_KEY` 와 `LANGFUSE_*` 가 모두 있어야 합니다. 문제를 풀고 나면 [cloud.langfuse.com](https://cloud.langfuse.com) 대시보드에 그 실행들이 그대로 쌓여 있습니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우
load_dotenv("../../../.env") # 교안 폴더 안의 과제/정답 에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

In [ ]:
# [제공 코드] 관측 콜백 준비 - 실행만 하세요.
# 핸들러는 .env 의 LANGFUSE_* 를 스스로 읽습니다 - 키를 인자로 넘기지 않습니다.
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

# 키를 먼저 확인합니다 - 핸들러를 만든 뒤에 검사하면 이 안내가 묻힙니다.
if not (os.getenv('LANGFUSE_PUBLIC_KEY') and os.getenv('LANGFUSE_SECRET_KEY')):
    raise RuntimeError('Langfuse 키를 찾지 못했습니다. 일차 폴더 .env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 를 채우고 커널을 재시작하세요.')

handlers = [CallbackHandler()]

# 키가 '있지만 틀린' 경우 langfuse 는 전송만 조용히 실패합니다 - 그래서 인증을 여기서 확인합니다.
try:
    authenticated = get_client().auth_check()
except Exception:
    authenticated = False
if not authenticated:
    raise RuntimeError('Langfuse 인증에 실패했습니다. 일차 폴더 .env 의 키와 LANGFUSE_BASE_URL 을 확인하고 커널을 재시작하세요.')

print('Langfuse 관측 켜짐 - 이제부터의 호출이 대시보드로 전송됩니다')

In [ ]:
# [제공 코드] 서버에서 기록을 되읽는 도우미 - 실행만 하세요.
# 전송에는 몇 초 지연이 있습니다. 교안 3.3 에서 본 '기다렸다 읽기'를 함수로 묶어 둔 것입니다.
import time

from langfuse import get_client

langfuse = get_client()


def fetch_generation(trace_id, usage, tries=15):
    """방금 그 호출의 기록이 서버에 붙을 때까지 기다렸다가 돌려준다.

    trace 껍데기가 먼저 오고 모델 호출 기록은 조금 늦게 붙는다. 토큰 수가 같은 기록을
    찾아 돌려주므로, 그 trace 에 다른 호출이 함께 들어 있어도 우리 호출을 집어낸다.
    """
    for attempt in range(tries):
        try:
            trace = langfuse.api.trace.get(trace_id)
            for observation in trace.observations:
                if (observation.type == 'GENERATION'
                        and observation.usage.input == usage['input_tokens']
                        and observation.usage.output == usage['output_tokens']
                        and observation.calculated_total_cost):
                    return observation
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError('기록이 서버에 보이지 않습니다. 대시보드에서 직접 확인해 보세요.')


def fetch_session(session_id, least, tries=10):
    """세션에 least 건 이상이 묶일 때까지 기다렸다 돌려준다."""
    for attempt in range(tries):
        try:
            session = langfuse.api.sessions.get(session_id)
            if len(session.traces) >= least:
                return session
        except Exception:
            pass
        time.sleep(3)
    raise RuntimeError('세션이 서버에 보이지 않습니다. flush() 를 불렀는지 확인하세요.')


In [ ]:
# [제공 코드] 집계 요약을 돌려주는 분석 도구입니다 - 실행만 하세요.
import pandas as pd


def summarize_by(df, group_col, value_col):
    """group_col 별 value_col 평균을 내림차순 문자열로 요약한다(리포트 입력용)."""
    s = df.groupby(group_col)[value_col].mean().sort_values(ascending=False).round(1)
    parts = [f"{k} {v}" for k, v in s.items()]
    return f"{group_col}별 평균 {value_col}: " + ", ".join(parts)


## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다
gym = pd.read_csv('../../data/gym_members.csv')
print('회원 수:', len(gym))
print(gym.head())
gym_summary = summarize_by(gym, '회원권', '월방문횟수')
print()
print(gym_summary)

## 1. 관측 콜백 붙이기
**배경**: 관측을 붙이는 자리는 호출 로직이 아니라 **`config` 한 곳**입니다.

**요구사항**:
- 함수 **`observed_ask(chat_model, question)`** 를 만드세요. 받은 모델로 질문을 부르되 준비 셀의 **`handlers`** 를 `config={'callbacks': handlers}` 로 함께 넘기고 **응답 객체를 그대로** 돌려줍니다.
- 그 함수를 `observed_ask(model, '헬스장 재등록률을 높이는 방법 한 가지')` 로 불러 응답을 **`cb_response`** 에, 그 `.text` 를 **`cb_answer`** 에 담으세요.
- 모델을 **인자로** 받는 이유는 자가채점이 **가짜 모델**을 넣어 `config` 를 정말 넘겼는지 보기 때문입니다(그 검사에서는 실제 호출이 일어나지 않습니다).

**확인 기준**: `cb_answer` 가 비어 있지 않고, 대시보드 **Tracing → Traces** 에 그 호출이 한 건 올라옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 받은 모델로 부르되, config 자리에 준비 셀의 handlers 를 실어 보낸다.

세부구현:
1. 함수는 모델과 질문을 받아 그 모델의 invoke 를 부른다.
2. invoke 에 config 를 함께 넘긴다 - 키는 'callbacks', 값은 준비 셀의 handlers 다.
3. 응답 객체를 그대로 돌려주고, 부른 쪽에서 .text 를 꺼내 cb_answer 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(handlers, list) and handlers, '준비 셀의 handlers 를 그대로 쓰세요'
assert isinstance(cb_answer, str) and len(cb_answer.strip()) >= 10, 'cb_answer 가 비어 있습니다'
assert cb_answer == cb_response.text, 'cb_answer 에는 cb_response 의 본문을 담으세요'
assert cb_response.usage_metadata['total_tokens'] > 0, (
    '실제 호출이라면 응답에 토큰 수가 함께 옵니다 - 문자열을 손으로 적으면 여기서 걸립니다')

# 콜백을 정말 넘겼는지는 결과만 봐서는 알 수 없다 - 가짜 모델을 넣어 config 를 들여다본다
from types import SimpleNamespace

seen = {}


def probe_invoke(question, **kwargs):
    seen['config'] = kwargs.get('config')
    return SimpleNamespace(text='가짜 응답', usage_metadata={'total_tokens': 1})


observed_ask(SimpleNamespace(invoke=probe_invoke), '점검용 질문')
assert seen.get('config'), 'observed_ask 안에서 invoke 에 config 를 함께 넘겨야 합니다'
assert seen['config'].get('callbacks') is handlers, (
    "config={'callbacks': handlers} 로 준비 셀의 handlers 를 그대로 넘기세요")
print('✅ 통과! (전송 여부는 대시보드에서 눈으로 확인하세요)')

## 2. 여러 호출의 토큰 모으기
**배경**: 한 요청이 모델을 여러 번 부르면 요금도 그만큼 늘어납니다. 그래서 **더해서** 봅니다.

**요구사항**:
- 아래 세 질문을 차례로 `model.invoke` 로 부르세요(콜백은 넣어도 되고 안 넣어도 됩니다).
  `['헬스장 회원권 종류를 한 줄로', '재등록률이 뭐야?', '헬스장 성수기는 언제야?']`
- 호출마다 응답의 `usage_metadata['total_tokens']` 를 **`token_log`** 리스트에 담고, 그 합을 **`total_tokens`** 에 담으세요.

**확인 기준**: `token_log` 는 길이 3 이고 값은 모두 0 보다 큽니다. `total_tokens` 는 그 셋의 합입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문 목록을 for 로 돌면서 응답의 토큰 수를 리스트에 담는다.

세부구현:
1. 빈 리스트 token_log 를 만든다.
2. 질문마다 invoke 하고 usage_metadata['total_tokens'] 를 append 한다.
3. sum(token_log) 을 total_tokens 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(token_log, list) and len(token_log) == 3, 'token_log 는 세 호출의 토큰 수 목록입니다'
assert all(isinstance(n, int) and n > 0 for n in token_log), f'토큰 수가 이상합니다: {token_log}'
assert total_tokens == sum(token_log), 'total_tokens 는 token_log 의 합이어야 합니다'
print('✅ 통과! 총', total_tokens, '토큰')

## 3. 공식 요금표로 비용 계산하기
**배경**: 토큰은 **100만 개 단위**로 값이 매겨지고 **입력과 출력의 단가가 다릅니다**.

**요구사항**:
- 100만 토큰당 단가를 담은 딕셔너리 **`PRICE_PER_1M`** 을 만드세요. 우리 모델(`gpt-4o-mini`)의 공식 단가는 **입력 $0.15 · 출력 $0.60** 입니다. 키는 `'input'` 과 `'output'` 을 쓰세요.
- 함수 **`cost_usd(usage)`** 를 만드세요. `usage_metadata` 를 받아 그 호출의 비용을 **달러(float)** 로 돌려줍니다.
- 1번에서 담아 둔 `cb_response.usage_metadata` 로 비용을 구해 **`cb_cost`** 에 담고 출력하세요.

**확인 기준**: 입력 1,000·출력 500 토큰이면 **$0.00045** 가 나와야 합니다(자가채점이 이 값으로 검산합니다). `cb_cost` 는 $0.0000x 수준의 아주 작은 값입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 비용 = 입력토큰/1000000*입력단가 + 출력토큰/1000000*출력단가

세부구현:
1. PRICE_PER_1M 은 {'input': 0.15, 'output': 0.60} 이다.
2. usage 에서 input_tokens·output_tokens 를 꺼내 각각 곱한 뒤 더한다.
3. f'{값:.8f}' 로 찍으면 지수 표기 대신 자리수로 읽힌다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert PRICE_PER_1M['input'] == 0.15 and PRICE_PER_1M['output'] == 0.60, (
    '공식 단가는 100만 토큰당 입력 0.15 · 출력 0.60 달러입니다')
# 알려진 값으로 검산한다 - 1000*0.15/1e6 + 500*0.60/1e6 = 0.00045
checked = cost_usd({'input_tokens': 1000, 'output_tokens': 500})
assert abs(checked - 0.00045) < 1e-12, f'입력 1000·출력 500 이면 0.00045 여야 합니다: {checked}'
assert cb_cost > 0, 'cb_cost 가 0 입니다. cb_response 의 usage_metadata 를 넘겼는지 확인하세요'
print('✅ 통과! 검산값', checked)

## 4. Langfuse 가 계산한 값과 맞춰 보기
**배경**: 서버도 같은 요금표로 비용을 계산해 둡니다. 두 값이 어긋나면 **단가표가 다르다**는 신호입니다.

**요구사항**:
- `langfuse.start_as_current_observation(as_type='span', name='과제 비용 확인')` 으로 감싼 채 `'헬스장 신규 회원을 늘리는 방법 한 가지'` 를 **콜백과 함께** 부르고, 응답을 **`checked_response`** 에 담으세요.
- 그 안에서 `langfuse.get_current_trace_id()` 로 **`trace_id`** 를 얻으세요.
- span 을 빠져나온 뒤 `langfuse.flush()` 를 부르고, 준비 셀의 **`fetch_generation(trace_id, checked_response.usage_metadata)`** 로 그 **모델 호출 기록**을 받아 **`server_cost`** 에 `.calculated_total_cost` 를, **`my_cost`** 에 `cost_usd(...)` 결과를 담으세요.
- 두 값을 함께 출력하세요.

**확인 기준**: 두 값의 차이가 **거의 0**(1억분의 1 달러 미만)입니다. 서버에 나타나기까지 몇 초 걸립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 3.3 과 같은 순서다. span 으로 감싸 trace id 를 얻고, flush 한 뒤 되읽는다.

세부구현:
1. with langfuse.start_as_current_observation(as_type='span', name='과제 비용 확인'):
2. 그 안에서 invoke 하고 trace_id = langfuse.get_current_trace_id() 를 얻는다.
3. with 를 빠져나와 langfuse.flush() 를 부른다.
4. generation = fetch_generation(trace_id, checked_response.usage_metadata) 로 받는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(trace_id, str) and len(trace_id) > 10, 'trace_id 를 얻지 못했습니다'
assert server_cost > 0, '서버 비용이 0 입니다. 콜백을 함께 넘겼는지 확인하세요'
assert abs(server_cost - my_cost) < 1e-8, (
    f'두 값이 다릅니다(서버 {server_cost} / 내 계산 {my_cost}). 단가표와 모델 이름을 확인하세요')
print('✅ 통과! 서버와 내 계산이 같습니다')

## 5. 프롬프트를 서버에 올리고 라벨로 불러오기
**배경**: 프롬프트를 코드 밖에 두면 코드를 다시 배포하지 않고 문구만 갈아 끼울 수 있습니다.

**요구사항**:
- `langfuse.create_prompt` 로 이름 **`'gym-report-writer'`** 프롬프트를 올리세요.
  - `prompt` 본문에는 채울 자리 **`{{summary}}`** 가 들어가야 합니다(예: `'너는 헬스장 데이터 분석가다. 아래 수치를 두 문장으로 요약하라.\n{{summary}}'`).
  - `labels=['production']` 을 함께 주세요.
- 올린 뒤 `langfuse.get_prompt('gym-report-writer', label='production', cache_ttl_seconds=0)` 로 되불러 **`live_prompt`** 에 담고, 그 `.version` 을 **`live_version`** 에 담으세요.
- `live_prompt.compile(summary=gym_summary)` 로 값을 채워 **`filled_prompt`** 에 담으세요.

**확인 기준**: `live_version` 은 1 이상의 정수이고, `filled_prompt` 안에 `gym_summary` 문자열이 그대로 들어 있습니다. 대시보드 **Prompts** 탭에도 그 이름이 보입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 올리는 것은 create_prompt, 꺼내는 것은 get_prompt 다. 둘 다 langfuse. 을 앞에 붙인다.

세부구현:
1. langfuse.create_prompt(name=..., prompt=..., labels=['production'])
2. cache_ttl_seconds=0 을 주면 방금 올린 버전을 바로 받는다.
3. compile(summary=gym_summary) 가 {{summary}} 자리를 채운 문자열을 돌려준다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(live_version, int) and live_version >= 1, f'버전이 이상합니다: {live_version}'
assert '{{summary}}' not in filled_prompt, 'compile 로 {{summary}} 자리를 채워야 합니다'
assert gym_summary in filled_prompt, 'filled_prompt 안에 gym_summary 가 그대로 들어 있어야 합니다'
print('✅ 통과! gym-report-writer v' + str(live_version), '- 대시보드 Prompts 탭에서도 확인해 보세요')

## 6. 대화 한 판을 세션으로 묶기
**배경**: 사용자가 이어서 여러 번 물으면 trace 도 여러 건입니다. 나중에 그 대화만 열어 보려면 묶어 두어야 합니다.

**요구사항**:
- 세션 이름표 **`session_id`** 를 `'gwaje-gym-session'` 으로 정하세요.
- 아래 두 질문을 **같은 이름표**로 부르세요. 이름표는 `config` 의 `metadata={'langfuse_session_id': session_id}` 로 넘깁니다(콜백도 함께).
  `['3개월 회원권을 추천하는 이유 한 가지', '방금 이유를 한 문장으로 줄여줘']`
- `langfuse.flush()` 를 부른 뒤 준비 셀의 **`fetch_session(session_id, 2)`** 로 세션을 받아 그 `.traces` 를 **`session_traces`** 에 담으세요.

**확인 기준**: `session_traces` 가 **2건 이상**입니다. 대시보드 **Tracing → Sessions** 에서 `gwaje-gym-session` 한 줄을 열면 두 호출이 함께 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 콜백과 같은 자리에 metadata 를 하나 더 실어 보낸다.

세부구현:
1. config={'callbacks': handlers, 'metadata': {'langfuse_session_id': session_id}}
2. 두 질문을 for 로 돌며 같은 config 로 부른다.
3. flush() 뒤 fetch_session(session_id, 2).traces 를 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert session_id == 'gwaje-gym-session', "세션 이름표는 'gwaje-gym-session' 입니다"
assert len(session_traces) >= 2, (
    f'세션에 두 건이 묶여야 합니다: {len(session_traces)}건. 두 호출에 같은 이름표를 줬는지 확인하세요')
print('✅ 통과! Sessions 탭에서', session_id, '를 열어 보세요')

In [ ]:
# [제공 코드] 라우팅 문제에서 쓸 도구와 도우미 - 실행만 하세요.
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool

FEES = {'1개월': 55000, '3개월': 135000, '6개월': 240000, '12개월': 420000}


def fee_of(plan):
    """'3개월 정기권' 처럼 이름이 섞여 들어와도 찾도록 부분 일치로 고른다."""
    for name, fee in FEES.items():
        if name in plan:
            return fee
    return 55000


def tool_trace(result):
    """메시지 기록에서 '어떤 도구가 불렸나'를 순서대로 뽑는다(교안 6절과 같은 도우미)."""
    names = []
    for message in result['messages']:
        # 도구 호출은 AIMessage 에만 담긴다(사람 말·도구 결과에는 없다) - 타입으로 가른다
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                names.append(call['name'])
    return names


@tool
def info(plan: str) -> str:
    """회원 정보를 조회한다."""    # ← 요금을 돌려주는 도구인데 그 말이 어디에도 없다
    return str(fee_of(plan))


@tool
def gym_guide(plan: str) -> str:
    """회원권 상품을 안내한다."""
    return f'{plan} 회원권은 헬스장 전 구역을 이용할 수 있는 상품입니다.'


FEE_QUESTION = '3개월 회원권 요금이 얼마인가요?'
print('도구 준비 완료:', [info.name, gym_guide.name])


## 7. 기록을 읽고 라우팅 고치기
**배경**: 답이 이상할 때 가장 먼저 볼 곳은 **어떤 도구가 불렸나**이고, 그다음 의심할 곳은 **도구의 이름과 설명**입니다.

준비 셀이 모호한 도구 한 쌍(`info`·`gym_guide`)과 도우미 `tool_trace` 를 줬습니다. 먼저 그대로 돌려 보고, **이름과 설명만** 고쳐 다시 돌립니다.

**요구사항**:
- `create_agent(model, [info, gym_guide])` 로 에이전트를 만들어 `FEE_QUESTION` 을 물으세요(`config` 에 콜백을 함께 넘깁니다). 결과의 도구 목록을 `tool_trace` 로 뽑아 **`vague_tools`** 에 담으세요. **여기서는 엉뚱한 도구가 불리는 것이 정상입니다.**
- 도구 **`membership_fee(plan: str) -> int`** 를 만들고 `@tool` 을 붙이세요. 안에서는 `fee_of(plan)` 을 그대로 돌려줍니다. **바꾸는 것은 이름과 docstring 뿐입니다**(무엇을 받아 무엇을 돌려주는지 한 문장으로).
- `create_agent(model, [membership_fee, gym_guide])` 로 다시 만들어 같은 질문을 묻고, 도구 목록을 **`fixed_tools`** 에, 마지막 답 본문을 **`fixed_answer`** 에 담으세요.

**확인 기준**: `fixed_tools` 에 `membership_fee` 가 들어 있고 `fixed_answer` 에 **135,000** 이 나옵니다. 두 실행 모두 대시보드에 남아 나중에 나란히 비교할 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 로직·질문·모델은 그대로 두고 도구의 이름과 설명만 고친다.

세부구현:
1. agent.invoke({'messages': FEE_QUESTION}, config={'callbacks': handlers})
2. tool_trace(결과) 로 불린 도구 이름 목록을 얻는다.
3. 답 본문은 결과['messages'][-1].text 다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(vague_tools, list), 'vague_tools 에는 tool_trace 가 돌려준 목록을 담으세요'
assert membership_fee.name == 'membership_fee', '도구 이름을 membership_fee 로 지으세요'
assert membership_fee.description and len(membership_fee.description.strip()) >= 15, (
    'docstring 이 비었거나 너무 짧습니다. 무엇을 받아 무엇을 돌려주는지 한 문장으로 적으세요')
assert membership_fee.invoke({'plan': '3개월'}) == 135000, (
    'membership_fee 는 fee_of 의 값을 그대로 돌려줘야 합니다')
assert 'membership_fee' in fixed_tools, (
    f'고친 뒤에도 요금 도구가 불리지 않았습니다: {fixed_tools}. docstring 을 더 분명하게 적어 보세요')
assert '135,000' in fixed_answer or '135000' in fixed_answer, (
    f'답에 3개월 요금이 들어 있지 않습니다: {fixed_answer}')
print('✅ 통과!  모호한 쪽', vague_tools, '→ 고친 쪽', fixed_tools)

---
## 다 풀었다면

[cloud.langfuse.com](https://cloud.langfuse.com) 을 열어 세 곳을 확인하세요.

- **Tracing → Traces**: 방금 푼 호출들이 그대로 쌓여 있습니다. 하나를 열어 토큰·지연·비용을 보세요.
- **Tracing → Sessions**: `gwaje-gym-session` 한 줄에 6번의 두 호출이 묶여 있습니다.
- **Prompts**: `gym-report-writer` 가 버전과 `production` 라벨을 달고 있습니다.

노트북을 닫아도 이 기록은 남습니다. **그것이 관측을 붙여 두는 이유입니다.**